# DSE ZG532 : Introduction to Data Science
## Assignment 1 (PS1) : Retail Sales, Customer Demographics, and Revenue Analysis

**Name: Hrithik Raj**

**BITS ID: 2026ND04219**

---

### Notebook Structure

| Stage | Task |
|---|---|
| Input, Ingestion, Validation | Task 1 |
| Cleaning and Transformation | Task 2 |
| Numerical Analysis | Task 3 |
| Extraction and Aggregation | Task 4 |
| Output: Insights and Conclusion | Task 5 |

---

## Task 1: Assigned Dataset, Python Fundamentals, and Data Ingestion

In [1]:
!pip install -q kagglehub

In [2]:
import os
import pathlib

import numpy as np
import pandas as pd
from scipy import stats

import kagglehub

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [3]:
def show_table(frame, caption):
    """
    Display a DataFrame as one clean table with a title and aligned headers.

    Parameters
    ----------
    frame : pandas.DataFrame
        Table to show.
    caption : str
        Title shown above the table.

    Returns
    -------
    None
    """
    table = frame.reset_index()
    first_col = table.columns[0]

    styled = (
        table.style
        .hide(axis="index")
        .set_caption(caption)
        .set_properties(subset=[first_col], **{"text-align": "left"})
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "15px"), ("font-weight", "bold"),
                       ("text-align", "left"), ("padding", "12px 0 8px 0")]},
            {"selector": "th",
             "props": [("text-align", "right"), ("padding", "6px 18px"),
                       ("border-bottom", "1px solid #555")]},
            {"selector": "th.col_heading.col0",
             "props": [("text-align", "left")]},
            {"selector": "td",
             "props": [("text-align", "right"), ("padding", "6px 18px")]},
            {"selector": "table",
             "props": [("margin-bottom", "28px"), ("border-collapse", "collapse")]},
        ])
    )
    display(styled)

In [4]:
# student
STUDENT = {
    "name": "Hrithik Raj",
    "bits_id": "2026ND04219",
    "course": "DSE ZG532 Introduction to Data Science",
    "assignment": "Assignment 1 PS1",
}

# source metadata
SOURCE_METADATA = {
    "dataset_title": "Retail Sales Dataset",
    "kaggle_publisher": "Mohammad Talib",
    "official_page": "https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset",
    "fixed_handle": "mohammadtalib786/retail-sales-dataset/versions/1",
    "required_filename": "retail_sales_dataset.csv",
    "licence": "CC0: Public Domain",
    "access_date": "2026-08-28",
    "domain": "Retail / transactional sales",
}

# expected structure
EXPECTED_SHAPE = (1000, 9)
REQUIRED_COLUMNS = [
    "Transaction ID", "Date", "Customer ID", "Gender", "Age",
    "Product Category", "Quantity", "Price per Unit", "Total Amount",
]

# category rules
CATEGORY_RULES = {
    "Gender": {"Female", "Male"},
    "Product Category": {"Beauty", "Clothing", "Electronics"},
}

# processing rules
PROCESSING_RULES = {
    "transaction_id_range": (1, 1000),
    "customer_id_prefix": "CUST",
    "customer_id_count": 1000,
    "age_range": (18, 100),
    "min_quantity": 1,
    "amount_tolerance": 0.01,
}

for key, value in SOURCE_METADATA.items():
    print(f"{key:20s}: {value}")

dataset_title       : Retail Sales Dataset
kaggle_publisher    : Mohammad Talib
official_page       : https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
fixed_handle        : mohammadtalib786/retail-sales-dataset/versions/1
required_filename   : retail_sales_dataset.csv
licence             : CC0: Public Domain
access_date         : 2026-08-28
domain              : Retail / transactional sales


In [5]:
CITATION_LABELS = {
    "dataset_title":     "Dataset title",
    "kaggle_publisher":  "Kaggle publisher",
    "official_page":     "Official Kaggle page",
    "fixed_handle":      "Fixed versioned handle",
    "required_filename": "Required file",
    "licence":           "Licence",
    "access_date":       "Access date",
    "domain":            "Domain",
}

citation = pd.DataFrame(
    [(label, SOURCE_METADATA[key]) for key, label in CITATION_LABELS.items()],
    columns=["Item", "Value"],
)
citation.loc[len(citation)] = [
    "Expected initial shape",
    f"{EXPECTED_SHAPE[0]} records x {EXPECTED_SHAPE[1]} columns",
]
citation = citation.set_index("Item")

show_table(citation, "Dataset citation")

Item,Value
Dataset title,Retail Sales Dataset
Kaggle publisher,Mohammad Talib
Official Kaggle page,https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
Fixed versioned handle,mohammadtalib786/retail-sales-dataset/versions/1
Required file,retail_sales_dataset.csv
Licence,CC0: Public Domain
Access date,2026-08-28
Domain,Retail / transactional sales
Expected initial shape,1000 records x 9 columns


### Assigned Practical Questions

**Q1:** Which product-category and gender combination generated the maximum total
revenue, and what were its transaction count, total units, and mean transaction
value? If two or more combinations share the maximum total revenue, report and
interpret every tied combination.

**Q2:** How did transaction count, total units, total revenue, and mean transaction
value vary by calendar month and customer age group?

**Q3:** How did transaction count, total units, and total revenue vary across
unit-price bands and product categories?

| Question | Supporting output |
|---|---|
| Q1 | product_category and gender groupby summary, Task 4 |
| Q2 | month_period and age_group summary, Task 4 |
| Q3 | unit_price_band and product_category summary, Task 4 |


In [6]:
def load_and_validate(dataset_handle, required_filename, expected_shape, required_columns):
    """
    Download the assigned Kaggle dataset version and validate its file, schema and shape.

    Parameters
    ----------
    dataset_handle : str
        Fixed versioned Kaggle handle, for example "owner/dataset/versions/1".
    required_filename : str
        Exact CSV filename that must be present in the downloaded folder.
    expected_shape : tuple of (int, int)
        Expected (rows, columns) of the raw file before cleaning.
    required_columns : list of str
        Exact column names the file is required to contain.

    Returns
    -------
    pandas.DataFrame
        The validated raw DataFrame.

    Raises
    ------
    FileNotFoundError
        If the required CSV is not present in the downloaded folder.
    ValueError
        If the column schema or the shape does not match the specification.
    """
    # download and build the exact path
    base_path = pathlib.Path(kagglehub.dataset_download(dataset_handle))
    csv_path = base_path / required_filename

    print("INGESTION")
    print(f"  Handle used     : {dataset_handle}")
    print(f"  Download folder : {base_path}")
    print(f"  Files available : {[p.name for p in base_path.iterdir()]}")

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Required file '{required_filename}' was not found in {base_path}."
        )

    # header and delimiter evidence
    with open(csv_path, "r", encoding="utf-8") as file_handle:
        raw_header = file_handle.readline().strip()
    field_count = raw_header.count(",") + 1

    print(f"  File loaded     : {csv_path}")
    print(f"  Raw header line : {raw_header}")
    print(f"  Delimiter       : ',' gives {field_count} header fields")

    df = pd.read_csv(csv_path)
    print(f"  Records loaded  : {len(df)}")

    # column check, loop plus set lookup
    actual_columns = list(df.columns)
    actual_lookup = set(actual_columns)

    missing_columns = []
    for column in required_columns:
        if column not in actual_lookup:
            missing_columns.append(column)

    unexpected_columns = sorted(actual_lookup - set(required_columns))
    shape_matches = df.shape == expected_shape

    print("\nVALIDATION")
    print(f"  Actual columns     : {actual_columns}")
    print(f"  Missing columns    : {missing_columns if missing_columns else 'None (0)'}")
    print(f"  Unexpected columns : {unexpected_columns if unexpected_columns else 'None (0)'}")
    print(f"  Shape found        : {df.shape}")
    print(f"  Shape expected     : {expected_shape}")
    print(f"  Shape matches      : {shape_matches}")

    # collect every failure, then stop once
    validation_checks = [
        (bool(missing_columns), f"missing columns {missing_columns}"),
        (bool(unexpected_columns), f"unexpected columns {unexpected_columns}"),
        (not shape_matches, f"shape {df.shape} does not match expected {expected_shape}"),
    ]
    validation_errors = [message for has_failed, message in validation_checks if has_failed]

    if validation_errors:
        raise ValueError("Validation FAILED: " + "; ".join(validation_errors))

    print("\n  RESULT: VALIDATION PASSED")
    return df

In [7]:
raw_df = load_and_validate(
    dataset_handle    = SOURCE_METADATA["fixed_handle"],
    required_filename = SOURCE_METADATA["required_filename"],
    expected_shape    = EXPECTED_SHAPE,
    required_columns  = REQUIRED_COLUMNS,
)

100%|██████████| 11.2k/11.2k [00:00<00:00, 9.63MB/s]

Extracting files...
INGESTION
  Handle used     : mohammadtalib786/retail-sales-dataset/versions/1
  Download folder : /root/.cache/kagglehub/datasets/mohammadtalib786/retail-sales-dataset/versions/1
  Files available : ['retail_sales_dataset.csv']
  File loaded     : /root/.cache/kagglehub/datasets/mohammadtalib786/retail-sales-dataset/versions/1/retail_sales_dataset.csv
  Raw header line : Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
  Delimiter       : ',' gives 9 header fields
  Records loaded  : 1000

VALIDATION
  Actual columns     : ['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']
  Missing columns    : None (0)
  Unexpected columns : None (0)
  Shape found        : (1000, 9)
  Shape expected     : (1000, 9)
  Shape matches      : True

  RESULT: VALIDATION PASSED


In [8]:
COLUMN_DESCRIPTIONS = {
    "Transaction ID":   "Unique number for each sale, from 1 to 1000.",
    "Date":             "Date the sale happened. Loaded as text, changed to a date in Task 2.",
    "Customer ID":      "Unique customer code, CUST001 to CUST1000.",
    "Gender":           "Customer gender. Only Female or Male.",
    "Age":              "Customer age in years, between 18 and 100.",
    "Product Category": "Product group bought: Beauty, Clothing or Electronics.",
    "Quantity":         "How many units were bought in the sale.",
    "Price per Unit":   "Price of one unit.",
    "Total Amount":     "Value of the sale. Should equal Quantity times Price per Unit.",
}

schema = pd.DataFrame(
    {
        "Dtype":       [str(raw_df[c].dtype) for c in REQUIRED_COLUMNS],
        "Non-null":    [int(raw_df[c].notna().sum()) for c in REQUIRED_COLUMNS],
        "Unique":      [int(raw_df[c].nunique()) for c in REQUIRED_COLUMNS],
        "Example":     [raw_df[c].iloc[0] for c in REQUIRED_COLUMNS],
        "Description": [COLUMN_DESCRIPTIONS[c] for c in REQUIRED_COLUMNS],
    },
    index=REQUIRED_COLUMNS,
)
schema.index.name = "Column"

show_table(schema, "Schema: column descriptions")
show_table(raw_df.head(10), "Representative records: first 10 rows")

Column,Dtype,Non-null,Unique,Example,Description
Transaction ID,int64,1000,1000,1,"Unique number for each sale, from 1 to 1000."
Date,object,1000,345,2023-11-24,"Date the sale happened. Loaded as text, changed to a date in Task 2."
Customer ID,object,1000,1000,CUST001,"Unique customer code, CUST001 to CUST1000."
Gender,object,1000,2,Male,Customer gender. Only Female or Male.
Age,int64,1000,47,34,"Customer age in years, between 18 and 100."
Product Category,object,1000,3,Beauty,"Product group bought: Beauty, Clothing or Electronics."
Quantity,int64,1000,4,3,How many units were bought in the sale.
Price per Unit,int64,1000,5,50,Price of one unit.
Total Amount,int64,1000,18,150,Value of the sale. Should equal Quantity times Price per Unit.


index,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100
5,6,2023-04-25,CUST006,Female,45,Beauty,1,30,30
6,7,2023-03-13,CUST007,Male,46,Clothing,2,25,50
7,8,2023-02-22,CUST008,Male,30,Electronics,4,25,100
8,9,2023-12-13,CUST009,Male,63,Electronics,2,300,600
9,10,2023-10-07,CUST010,Female,52,Clothing,4,50,200


In [9]:
NL = "\n"
ROWS = len(raw_df)

dtype_lines = NL.join(f"   {c:18s}: {raw_df[c].dtype}" for c in REQUIRED_COLUMNS)

unique_lines = NL.join(f"   {c:18s}: {raw_df[c].nunique():>5d}" for c in REQUIRED_COLUMNS)

id_lines = NL.join(
    f"   {c:18s}: {raw_df[c].nunique()} unique / {ROWS} rows, "
    f"{'all distinct' if raw_df[c].nunique() == ROWS else 'DUPLICATES PRESENT'}"
    for c in ["Transaction ID", "Customer ID"]
)

category_lines = NL.join(
    f"   {c:18s}: {sorted(raw_df[c].unique())}" for c in ["Gender", "Product Category"]
)

price_lines = NL.join(
    f"      {price:>6} : {count:>4d} transactions"
    for price, count in raw_df["Price per Unit"].value_counts().sort_index().items()
)

print(f"""OBSERVATIONS

1. Shape
   Rows x Columns    : {raw_df.shape}
   Expected          : {EXPECTED_SHAPE}
   Match             : {raw_df.shape == EXPECTED_SHAPE}

2. Data types
{dtype_lines}

3. Unique values per column
{unique_lines}

4. Identifier uniqueness
{id_lines}

5. Missing values
   Total missing cells  : {int(raw_df.isna().sum().sum())}
   Columns with missing : {int((raw_df.isna().sum() > 0).sum())}

6. Category labels found
{category_lines}

7. Price per Unit distribution
   Distinct values   : {raw_df['Price per Unit'].nunique()}
   Values            : {sorted(raw_df['Price per Unit'].unique())}
   Breakdown:
{price_lines}

8. Memory usage
   Deep memory usage : {raw_df.memory_usage(deep=True).sum() / 1024:.1f} KB
""")

OBSERVATIONS

1. Shape
   Rows x Columns    : (1000, 9)
   Expected          : (1000, 9)
   Match             : True

2. Data types
   Transaction ID    : int64
   Date              : object
   Customer ID       : object
   Gender            : object
   Age               : int64
   Product Category  : object
   Quantity          : int64
   Price per Unit    : int64
   Total Amount      : int64

3. Unique values per column
   Transaction ID    :  1000
   Date              :   345
   Customer ID       :  1000
   Gender            :     2
   Age               :    47
   Product Category  :     3
   Quantity          :     4
   Price per Unit    :     5
   Total Amount      :    18

4. Identifier uniqueness
   Transaction ID    : 1000 unique / 1000 rows, all distinct
   Customer ID       : 1000 unique / 1000 rows, all distinct

5. Missing values
   Total missing cells  : 0
   Columns with missing : 0

6. Category labels found
   Gender            : ['Female', 'Male']
   Product Category  :

### Structural Observations and Suitability

- Loaded exactly 1,000 rows and 9 columns, as required.
- `Date` loaded as text, so it needs parsing before any month grouping.
- `Gender` (2 labels) and `Product Category` (3 labels) suit the `category` dtype.
- `Transaction ID` and `Customer ID` are both 1000 unique across 1000 rows.
- No missing values anywhere.
- `Price per Unit` has only 5 distinct values (25, 30, 50, 300, 500), so the Q3 price bands will be coarse. Noted as a limitation.
- 260 KB in memory, so no sampling or chunked reading is needed.

**Suitable for Tasks 2-5: yes.** All 9 columns are present, the four numeric columns are complete, and every grouping key the three questions need (category, gender, date, age, price) is available. The formal validity checks are run in Task 2.


---

## Task 2: Data Cleaning and Transformation Using Pandas

In [10]:
def assess_quality(df, cols, label):
    """
    Check the data for problems and return the results as a table.

    Parameters
    ----------
    df : pandas.DataFrame
        Data to check.
    cols : dict
        Real column name for each short name, so one function serves raw and clean data.
    label : str
        Heading for the results column.

    Returns
    -------
    pandas.DataFrame
        One row per check with the count found.
    """
    # column names
    id_col       = cols["transaction_id"]
    date_col     = cols["date"]
    cust_col     = cols["customer_id"]
    gender_col   = cols["gender"]
    age_col      = cols["age"]
    category_col = cols["product_category"]
    qty_col      = cols["quantity"]
    price_col    = cols["price_per_unit"]
    amount_col   = cols["total_amount"]

    # rules
    smallest_id, biggest_id   = PROCESSING_RULES["transaction_id_range"]
    smallest_age, biggest_age = PROCESSING_RULES["age_range"]
    smallest_qty              = PROCESSING_RULES["min_quantity"]
    allowed_gap               = PROCESSING_RULES["amount_tolerance"]
    how_many_customers        = PROCESSING_RULES["customer_id_count"]
    prefix                    = PROCESSING_RULES["customer_id_prefix"]

    # valid customer ids
    good_customer_ids = set()
    for number in range(1, how_many_customers + 1):
        good_customer_ids.add(f"{prefix}{number:03d}")

    # bad dates
    real_dates = pd.to_datetime(df[date_col], errors="coerce")
    bad_dates = int(real_dates.isna().sum())

    # extra spaces
    extra_spaces = 0
    for column in [cust_col, gender_col, category_col]:
        before = df[column].astype(str)
        after = before.str.strip()
        extra_spaces = extra_spaces + int((before != after).sum())

    # total = qty x price
    should_be = df[qty_col] * df[price_col]
    is_correct = np.isclose(should_be, df[amount_col], atol=allowed_gap)
    wrong_totals = int((~is_correct).sum())

    # allowed labels
    gender_text   = df[gender_col].astype(str).str.strip().str.title()
    category_text = df[category_col].astype(str).str.strip().str.title()
    customer_text = df[cust_col].astype(str).str.strip()

    wrong_gender   = int((~gender_text.isin(CATEGORY_RULES["Gender"])).sum())
    wrong_category = int((~category_text.isin(CATEGORY_RULES["Product Category"])).sum())
    wrong_customer = int((~customer_text.isin(good_customer_ids)).sum())

    # counts
    rows           = len(df)
    columns        = df.shape[1]
    missing_values = int(df.isna().sum().sum())
    same_rows      = int(df.duplicated().sum())

    repeated_ids       = rows - df[id_col].nunique()
    repeated_customers = rows - df[cust_col].nunique()

    # ranges
    id_out_of_range  = int((~df[id_col].between(smallest_id, biggest_id)).sum())
    age_out_of_range = int((~df[age_col].between(smallest_age, biggest_age)).sum())
    small_quantity   = int((df[qty_col] < smallest_qty).sum())
    minus_price      = int((df[price_col] < 0).sum())
    minus_amount     = int((df[amount_col] < 0).sum())

    # results
    checks = [
        ("Rows",                                   rows),
        ("Columns",                                columns),
        ("Missing values",                         missing_values),
        ("Rows that are exact copies",             same_rows),
        ("Dates that cannot be read",              bad_dates),
        ("Transaction IDs that repeat",            repeated_ids),
        ("Transaction IDs not between 1 and 1000", id_out_of_range),
        ("Customer IDs that repeat",               repeated_customers),
        ("Customer IDs not in CUST001-CUST1000",   wrong_customer),
        ("Gender not Female or Male",              wrong_gender),
        ("Category not one of the 3 allowed",      wrong_category),
        ("Text with extra spaces",                 extra_spaces),
        ("Age not between 18 and 100",             age_out_of_range),
        ("Quantity less than 1",                   small_quantity),
        ("Price below zero",                       minus_price),
        ("Total below zero",                       minus_amount),
        ("Total is not Quantity times Price",      wrong_totals),
    ]

    report = pd.DataFrame(checks, columns=["Check", label])
    return report.set_index("Check")


# raw file uses spaced names, clean data uses short names
RAW_COLS = {
    "transaction_id":   "Transaction ID",
    "date":             "Date",
    "customer_id":      "Customer ID",
    "gender":           "Gender",
    "age":              "Age",
    "product_category": "Product Category",
    "quantity":         "Quantity",
    "price_per_unit":   "Price per Unit",
    "total_amount":     "Total Amount",
}

CLEAN_COLS = {}
for short_name in RAW_COLS:
    CLEAN_COLS[short_name] = short_name

In [11]:
before_report = assess_quality(raw_df, RAW_COLS, "Before cleaning")

# types, missing and unique in one table
column_summary = pd.DataFrame({
    "Type": raw_df.dtypes.astype(str),
    "Missing": raw_df.isna().sum(),
    "Unique": raw_df.nunique(),
})
column_summary.index.name = "Column"

show_table(before_report, "Before cleaning: data quality report")
show_table(column_summary, "Before cleaning: type, missing and unique per column")

Check,Before cleaning
Rows,1000
Columns,9
Missing values,0
Rows that are exact copies,0
Dates that cannot be read,0
Transaction IDs that repeat,0
Transaction IDs not between 1 and 1000,0
Customer IDs that repeat,0
Customer IDs not in CUST001-CUST1000,0
Gender not Female or Male,0


Column,Type,Missing,Unique
Transaction ID,int64,0,1000
Date,object,0,345
Customer ID,object,0,1000
Gender,object,0,2
Age,int64,0,47
Product Category,object,0,3
Quantity,int64,0,4
Price per Unit,int64,0,5
Total Amount,int64,0,18


In [12]:
RENAME_MAP = {
    "Transaction ID":   "transaction_id",
    "Date":             "date",
    "Customer ID":      "customer_id",
    "Gender":           "gender",
    "Age":              "age",
    "Product Category": "product_category",
    "Quantity":         "quantity",
    "Price per Unit":   "price_per_unit",
    "Total Amount":     "total_amount",
}

AGE_BINS     = [18, 30, 45, 60, 100]
AGE_LABELS   = ["18-30", "31-45", "46-60", "61+"]
PRICE_BINS   = [0, 50, 300, float("inf")]
PRICE_LABELS = ["Low", "Standard", "Premium"]

NEW_COLUMNS = ["month_period", "age_group", "unit_price_band"]

In [13]:
def clean_and_transform(df, rename_map, age_bins, age_labels, price_bins, price_labels):
    """
    Clean the raw data and add the three required new columns.

    Parameters
    ----------
    df : pandas.DataFrame
        Raw data from Task 1.
    rename_map : dict
        Old column name to new column name.
    age_bins, age_labels : list
        Cut points and names for the age groups.
    price_bins, price_labels : list
        Cut points and names for the price bands.

    Returns
    -------
    pandas.DataFrame
        Cleaned data with the three new columns added.
    """
    # copy, so raw data stays the same
    clean = df.copy()

    # new column names
    clean = clean.rename(columns=rename_map)

    # text to date
    clean["date"] = pd.to_datetime(clean["date"])

    # tidy text
    clean["customer_id"] = clean["customer_id"].str.strip()
    clean["gender"] = clean["gender"].str.strip().str.title()
    clean["product_category"] = clean["product_category"].str.strip().str.title()

    # category type
    clean["gender"] = clean["gender"].astype("category")
    clean["product_category"] = clean["product_category"].astype("category")

    # month
    clean["month_period"] = clean["date"].dt.to_period("M")

    # age group
    clean["age_group"] = pd.cut(
        clean["age"],
        bins=age_bins,
        labels=age_labels,
        include_lowest=True,
        right=True,
        ordered=True,
    )

    # price band
    clean["unit_price_band"] = pd.cut(
        clean["price_per_unit"],
        bins=price_bins,
        labels=price_labels,
        include_lowest=True,
        right=True,
        ordered=True,
    )

    # nothing may fall outside a band silently
    print("Rows with no age group  :", int(clean["age_group"].isna().sum()))
    print("Rows with no price band :", int(clean["unit_price_band"].isna().sum()))

    return clean

In [14]:
clean_df = clean_and_transform(
    df           = raw_df,
    rename_map   = RENAME_MAP,
    age_bins     = AGE_BINS,
    age_labels   = AGE_LABELS,
    price_bins   = PRICE_BINS,
    price_labels = PRICE_LABELS,
)

# overview
overview = pd.DataFrame(
    {"Value": [clean_df.shape[0], clean_df.shape[1], len(RENAME_MAP), len(NEW_COLUMNS)]},
    index=["Rows", "Columns", "Original columns", "New columns"],
)
overview.index.name = "Item"

# new column details
new_summary = pd.DataFrame(
    {
        "Type":    [str(clean_df[c].dtype) for c in NEW_COLUMNS],
        "Groups":  [clean_df[c].nunique() for c in NEW_COLUMNS],
        "Example": [str(clean_df[c].iloc[0]) for c in NEW_COLUMNS],
    },
    index=NEW_COLUMNS,
)
new_summary.index.name = "Column"

show_table(overview, "Cleaned data: overview")
show_table(clean_df.head().T, "Cleaned data: first 5 rows")
show_table(new_summary, "New columns: type, groups, example")

for column in NEW_COLUMNS:
    counts = clean_df[column].value_counts().sort_index().to_frame("Transactions")
    counts.index.name = column
    show_table(counts, f"Transactions per group: {column}")

Rows with no age group  : 0
Rows with no price band : 0


Item,Value
Rows,1000
Columns,12
Original columns,9
New columns,3


Column,0,1,2,3,4
transaction_id,1,2,3,4,5
date,2023-11-24 00:00:00,2023-02-27 00:00:00,2023-01-13 00:00:00,2023-05-21 00:00:00,2023-05-06 00:00:00
customer_id,CUST001,CUST002,CUST003,CUST004,CUST005
gender,Male,Female,Male,Male,Male
age,34,26,50,37,30
product_category,Beauty,Clothing,Electronics,Clothing,Beauty
quantity,3,2,1,1,2
price_per_unit,50,500,30,500,50
total_amount,150,1000,30,500,100
month_period,2023-11,2023-02,2023-01,2023-05,2023-05


Column,Type,Groups,Example
month_period,period[M],13,2023-11
age_group,category,4,31-45
unit_price_band,category,3,Low


month_period,Transactions
2023-01,76
2023-02,85
2023-03,73
2023-04,86
2023-05,105
2023-06,77
2023-07,72
2023-08,94
2023-09,65
2023-10,96


age_group,Transactions
18-30,273
31-45,303
46-60,331
61+,93


unit_price_band,Transactions
Low,604
Standard,197
Premium,199


In [15]:
after_report = assess_quality(clean_df, CLEAN_COLS, "After cleaning")

comparison = before_report.join(after_report)
comparison["Changed"] = comparison["Before cleaning"] != comparison["After cleaning"]

structure = pd.DataFrame(
    {
        "Before cleaning": [
            raw_df.shape[0], raw_df.shape[1],
            int(raw_df.isna().sum().sum()), int(raw_df.duplicated().sum()),
            str(raw_df["Date"].dtype), str(raw_df["Gender"].dtype),
        ],
        "After cleaning": [
            clean_df.shape[0], clean_df.shape[1],
            int(clean_df.isna().sum().sum()), int(clean_df.duplicated().sum()),
            str(clean_df["date"].dtype), str(clean_df["gender"].dtype),
        ],
    },
    index=["Rows", "Columns", "Missing values", "Duplicate rows", "Date type", "Gender type"],
)
structure.index.name = "Item"

show_table(comparison, "Data quality before and after cleaning")
show_table(structure, "Structure before and after cleaning")

Check,Before cleaning,After cleaning,Changed
Rows,1000,1000,False
Columns,9,12,True
Missing values,0,0,False
Rows that are exact copies,0,0,False
Dates that cannot be read,0,0,False
Transaction IDs that repeat,0,0,False
Transaction IDs not between 1 and 1000,0,0,False
Customer IDs that repeat,0,0,False
Customer IDs not in CUST001-CUST1000,0,0,False
Gender not Female or Male,0,0,False


Item,Before cleaning,After cleaning
Rows,1000,1000
Columns,9,12
Missing values,0,0
Duplicate rows,0,0
Date type,object,datetime64[ns]
Gender type,object,category


### Transformation operations used

| Operation | Where | Why it suits this data |
|---|---|---|
| `.str.strip()` | customer_id, gender, product_category | Removes stray spaces so labels group correctly. |
| `.str.title()` | gender, product_category | One spelling, so "male" and "Male" cannot split into two groups. |
| `.astype("category")` | gender, product_category | Only 2 and 3 repeated labels, so it saves memory and fixes the order. |
| `.dt.to_period("M")` | date | Drops the day so all sales in a month group together, which Q2 needs. |
| `pd.cut()` | age, price_per_unit | Turns numbers into a few named bands, which Q2 and Q3 need. |

### Derived column rules

| Column | Rule |
|---|---|
| `month_period` | `date.dt.to_period("M")`, shown as YYYY-MM |
| `age_group` | `pd.cut` on age, bins 18/30/45/60/100, labels 18-30, 31-45, 46-60, 61+ |
| `unit_price_band` | `pd.cut` on price_per_unit, bins 0/50/300/inf, labels Low, Standard, Premium |

All three use `include_lowest=True`, `right=True`, `ordered=True`.

### Cleaning decisions

Every quality check returned 0, so no rows were dropped and no values replaced. All 1000 rows kept.

- Missing values and exact duplicate rows: 0 found, nothing to do.
- Transaction ID and Customer ID: checked for repeats first, both unique, so neither was altered.
- Gender and Product Category: already valid, but strip and title case were still applied so the result does not depend on tidy input.
- Age, Quantity, Price per Unit, Total Amount: all in range, and Total Amount = Quantity x Price per Unit on all 1000 rows within 0.01.
- Shape goes from (1000, 9) to (1000, 12) because of the three new columns.

**Note:** the data runs 2023-01 to 2024-01, so 13 months. January 2024 has only 2 transactions and is a partial month, not a drop in sales.


---

## Task 3: Numerical Data Handling Using NumPy and SciPy

In [16]:
NUMERIC_COLUMNS = ["age", "quantity", "price_per_unit", "total_amount"]

# arrays
arrays = {}
for column in NUMERIC_COLUMNS:
    arrays[column] = clean_df[column].to_numpy(dtype=np.float64)

# properties
properties = pd.DataFrame(
    {
        "Shape":      [str(arrays[c].shape) for c in NUMERIC_COLUMNS],
        "Dimensions": [arrays[c].ndim for c in NUMERIC_COLUMNS],
        "Dtype":      [str(arrays[c].dtype) for c in NUMERIC_COLUMNS],
        "NaN count":  [int(np.isnan(arrays[c]).sum()) for c in NUMERIC_COLUMNS],
    },
    index=NUMERIC_COLUMNS,
)
properties.index.name = "Array"

show_table(properties, "NumPy arrays: shape, dimensions, type, NaN count")

Array,Shape,Dimensions,Dtype,NaN count
age,"(1000,)",1,float64,0
quantity,"(1000,)",1,float64,0
price_per_unit,"(1000,)",1,float64,0
total_amount,"(1000,)",1,float64,0


In [17]:
age    = arrays["age"]
qty    = arrays["quantity"]
price  = arrays["price_per_unit"]
amount = arrays["total_amount"]

# indexing and slicing
print("first value     :", amount[0])
print("first five      :", amount[:5])
print("last three      :", amount[-3:])

# boolean masking
big_sales = amount[amount > 1000]
print("sales above 1000:", big_sales.size)
print("their mean      :", round(big_sales.mean(), 2))

# vectorised op 1: element by element multiply
line_total = qty * price

# vectorised op 2: distance from the mean, broadcast
centred_amount = amount - amount.mean()

print("op 1 first five :", line_total[:5])
print("op 2 first five :", centred_amount[:5].round(2))

first value     : 150.0
first five      : [ 150. 1000.   30.  500.  100.]
last three      : [100. 150. 120.]
sales above 1000: 153
their mean      : 1554.25
op 1 first five : [ 150. 1000.   30.  500.  100.]
op 2 first five : [-306.  544. -426.   44. -356.]


In [18]:
# stats per column, ddof=0 for population values, NaN handled explicitly
rows = {}
for column in NUMERIC_COLUMNS:
    values = arrays[column]
    rows[column] = {
        "min":          np.nanmin(values),
        "max":          np.nanmax(values),
        "mean":         np.nanmean(values),
        "median":       np.nanmedian(values),
        "std (ddof=0)": np.nanstd(values, ddof=0),
        "var (ddof=0)": np.nanvar(values, ddof=0),
        "p25":          np.nanpercentile(values, 25),
        "p50":          np.nanpercentile(values, 50),
        "p75":          np.nanpercentile(values, 75),
    }

stats_table = pd.DataFrame(rows).T.round(2)
stats_table.index.name = "Column"

show_table(stats_table, "Summary statistics for every numeric column")

Column,min,max,mean,median,std (ddof=0),var (ddof=0),p25,p50,p75
age,18.000000,64.000000,41.390000,42.000000,13.670000,186.990000,29.000000,42.000000,53.000000
quantity,1.000000,4.000000,2.510000,3.000000,1.130000,1.280000,1.000000,3.000000,4.000000
price_per_unit,25.000000,500.000000,179.890000,50.000000,189.590000,35943.040000,30.000000,50.000000,300.000000
total_amount,25.000000,2000.000000,456.000000,135.000000,559.720000,313283.750000,60.000000,135.000000,900.000000


In [19]:
# required derived array, stays outside the DataFrame
recalculated_amount = qty * price

matches = np.isclose(recalculated_amount, amount, atol=0.01)
mismatches = int((~matches).sum())

sample = pd.DataFrame(
    {
        "quantity":            qty[:5],
        "price_per_unit":      price[:5],
        "recalculated_amount": recalculated_amount[:5],
        "total_amount":        amount[:5],
        "close (atol=0.01)":   matches[:5],
    }
)
sample.index.name = "Row"

show_table(sample, "recalculated_amount against total_amount, first 5 rows")
print("Rows compared :", amount.size)
print("Mismatches    :", mismatches)

Row,quantity,price_per_unit,recalculated_amount,total_amount,close (atol=0.01)
0,3.000000,50.000000,150.000000,150.000000,True
1,2.000000,500.000000,1000.000000,1000.000000,True
2,1.000000,30.000000,30.000000,30.000000,True
3,1.000000,500.000000,500.000000,500.000000,True
4,2.000000,50.000000,100.000000,100.000000,True


Rows compared : 1000
Mismatches    : 0


In [20]:
described = stats.describe(amount, ddof=0, nan_policy="omit")

scipy_table = pd.DataFrame(
    {
        "Value": [
            described.nobs,
            round(described.minmax[0], 2),
            round(described.minmax[1], 2),
            round(described.mean, 2),
            round(described.variance, 2),
            round(described.skewness, 4),
            round(described.kurtosis, 4),
        ]
    },
    index=["count", "min", "max", "mean", "variance (ddof=0)", "skewness", "kurtosis"],
)
scipy_table.index.name = "Statistic"

show_table(scipy_table, "scipy.stats.describe on total_amount")
print("NumPy variance (ddof=0) :", round(np.nanvar(amount, ddof=0), 2))
print("SciPy variance (ddof=0) :", round(described.variance, 2))

Statistic,Value
count,1000.000000
min,25.000000
max,2000.000000
mean,456.000000
variance (ddof=0),313283.750000
skewness,1.374100
kurtosis,0.805000


NumPy variance (ddof=0) : 313283.75
SciPy variance (ddof=0) : 313283.75


### What the numbers show

1. `total_amount` has mean 456.00 but median only 135.00. The mean sits more than
   three times the median, so a small group of large sales pulls the average up.
   SciPy reports skewness 1.3741, which confirms the long right tail.

2. The middle half of sales sits between p25 60.00 and p75 900.00. That spread is
   very wide, and it comes from the price structure: unit prices are only 25, 30,
   50, 300 and 500, so sales fall into a cheap cluster and an expensive cluster
   with nothing in between.

3. `recalculated_amount` matched `total_amount` for all 1000 rows within 0.01, so
   the Total Amount column is internally consistent and can be trusted for the
   revenue totals in Task 4.

4. `age` runs from 18 to 64 with mean 41.39 and median 42.00. Every customer is
   inside the allowed 18 to 100 range, but the real maximum is only 64. My 61+
   band therefore covers just 61 to 64, which is much narrower than its name
   suggests. I note this as a limitation.

5. Boolean masking found 153 sales above 1000, with mean 1554.25. Those 153 rows
   are 15.3 percent of transactions but they are what drags the mean away from
   the median.

6. I chose `scipy.stats.describe` with `ddof=0` because its variance then matches
   the population variance computed with NumPy. Both report 313283.75, which
   confirms the two calculations agree. I did not choose `zscore`, because
   standardised scores are mainly useful for removing outliers, and outlier
   removal is out of scope for this assignment.

---

## Task 4: Data Extraction and Aggregation

In [21]:
median_amount = clean_df["total_amount"].median()

# 1. loc with column selection and two conditions
extract_loc = clean_df.loc[
    (clean_df["total_amount"] > median_amount) & (clean_df["quantity"] >= 3),
    ["date", "transaction_id", "gender", "age_group", "product_category",
     "quantity", "price_per_unit", "total_amount"],
].sort_values(["total_amount", "quantity"], ascending=[False, False])

# 2. query
extract_query = clean_df.query(
    "31 <= age <= 45 and product_category in ['Clothing', 'Electronics']"
)[
    ["date", "customer_id", "gender", "age", "product_category",
     "quantity", "price_per_unit", "total_amount"]
].sort_values(["date", "total_amount"], ascending=[True, False])

# 3. iloc row and column slicing on a date sorted subset
chrono = clean_df.sort_values("date")[
    ["date", "transaction_id", "customer_id", "product_category",
     "quantity", "price_per_unit", "total_amount"]
]
extract_iloc = chrono.iloc[-10:, :7]

record_counts = pd.DataFrame(
    {"Records": [len(extract_loc), len(extract_query), len(extract_iloc)]},
    index=[
        "loc: above median and quantity >= 3",
        "query: age 31-45, Clothing or Electronics",
        "iloc: last 10 rows, first 7 columns",
    ],
)
record_counts.index.name = "Extraction"

print("Median total_amount:", median_amount)
show_table(record_counts, "Records returned by each extraction")
show_table(extract_loc.head(10), "Extraction 1 (loc): top 10 by total_amount")
show_table(extract_query.head(10), "Extraction 2 (query): first 10 by date")
show_table(extract_iloc, "Extraction 3 (iloc): final 10 rows")

Median total_amount: 135.0


Extraction,Records
loc: above median and quantity >= 3,319
"query: age 31-45, Clothing or Electronics",219
"iloc: last 10 rows, first 7 columns",10


index,date,transaction_id,gender,age_group,product_category,quantity,price_per_unit,total_amount
14,2023-01-16 00:00:00,15,Female,31-45,Electronics,4,500,2000
64,2023-12-05 00:00:00,65,Male,46-60,Electronics,4,500,2000
71,2023-05-23 00:00:00,72,Female,18-30,Electronics,4,500,2000
73,2023-11-22 00:00:00,74,Female,18-30,Beauty,4,500,2000
88,2023-10-01 00:00:00,89,Female,46-60,Electronics,4,500,2000
92,2023-07-14 00:00:00,93,Female,31-45,Beauty,4,500,2000
108,2023-10-18 00:00:00,109,Female,31-45,Electronics,4,500,2000
117,2023-05-16 00:00:00,118,Female,18-30,Electronics,4,500,2000
123,2023-10-27 00:00:00,124,Male,31-45,Clothing,4,500,2000
138,2023-12-15 00:00:00,139,Male,31-45,Beauty,4,500,2000


index,date,customer_id,gender,age,product_category,quantity,price_per_unit,total_amount
558,2023-01-01 00:00:00,CUST559,Female,40,Clothing,4,300,1200
179,2023-01-01 00:00:00,CUST180,Male,41,Clothing,3,300,900
420,2023-01-02 00:00:00,CUST421,Female,37,Clothing,3,500,1500
796,2023-01-07 00:00:00,CUST797,Male,40,Clothing,3,25,75
906,2023-01-08 00:00:00,CUST907,Female,45,Electronics,1,25,25
183,2023-01-10 00:00:00,CUST184,Male,31,Electronics,4,50,200
745,2023-01-11 00:00:00,CUST746,Female,33,Clothing,3,30,90
463,2023-01-13 00:00:00,CUST464,Male,38,Electronics,2,300,600
14,2023-01-16 00:00:00,CUST015,Female,42,Electronics,4,500,2000
437,2023-01-19 00:00:00,CUST438,Female,42,Clothing,1,30,30


index,date,transaction_id,customer_id,product_category,quantity,price_per_unit,total_amount
988,2023-12-28 00:00:00,989,CUST989,Electronics,1,25,25
663,2023-12-28 00:00:00,664,CUST664,Clothing,4,500,2000
428,2023-12-28 00:00:00,429,CUST429,Electronics,2,25,50
907,2023-12-29 00:00:00,908,CUST908,Beauty,4,300,1200
519,2023-12-29 00:00:00,520,CUST520,Electronics,4,25,100
232,2023-12-29 00:00:00,233,CUST233,Beauty,2,300,600
804,2023-12-29 00:00:00,805,CUST805,Beauty,3,500,1500
856,2023-12-31 00:00:00,857,CUST857,Electronics,2,25,50
210,2024-01-01 00:00:00,211,CUST211,Beauty,3,500,1500
649,2024-01-01 00:00:00,650,CUST650,Electronics,1,30,30


In [22]:
# one aggregation spec, reused for every summary
AGG_BASE = {
    "transaction_count":      ("transaction_id", "size"),
    "total_units":            ("quantity", "sum"),
    "total_revenue":          ("total_amount", "sum"),
    "mean_transaction_value": ("total_amount", "mean"),
}
AGG_WITH_MEDIAN = dict(AGG_BASE)
AGG_WITH_MEDIAN["median_transaction_value"] = ("total_amount", "median")


def summarise(df, keys, spec):
    """
    Group by keys and apply the named aggregations.

    Parameters
    ----------
    df : pandas.DataFrame
        Cleaned data.
    keys : list of str
        Columns to group by.
    spec : dict
        Named aggregations for groupby.agg.

    Returns
    -------
    pandas.DataFrame
        Grouped summary.
    """
    return df.groupby(keys, observed=True).agg(**spec)


# Q1: product category and gender
summary_cat_gender = (
    summarise(clean_df, ["product_category", "gender"], AGG_WITH_MEDIAN)
    .sort_values("total_revenue", ascending=False)
    .round(2)
)

# Q2: month and age group, chronological and band order kept
summary_month_age = (
    summarise(clean_df, ["month_period", "age_group"], AGG_BASE)
    .sort_index()
    .round(2)
)

# Q3: price band and category, band order kept, revenue sorted inside each band
summary_band_cat = (
    summarise(clean_df, ["unit_price_band", "product_category"], AGG_BASE)
    .sort_values(["unit_price_band", "total_revenue"], ascending=[True, False])
    .round(2)
)

show_table(summary_cat_gender, "Q1: product category and gender")
show_table(summary_month_age, "Q2: calendar month and age group")
show_table(summary_band_cat, "Q3: unit price band and product category")

product_category,gender,transaction_count,total_units,total_revenue,mean_transaction_value,median_transaction_value
Clothing,Female,174,441,81275,467.100000,120.000000
Electronics,Male,172,410,80170,466.100000,120.000000
Electronics,Female,170,439,76735,451.380000,150.000000
Beauty,Female,166,418,74830,450.780000,150.000000
Clothing,Male,177,453,74305,419.800000,150.000000
Beauty,Male,141,353,68685,487.130000,120.000000


month_period,age_group,transaction_count,total_units,total_revenue,mean_transaction_value
2023-01,18-30,25,68,8180,327.200000
2023-01,31-45,20,53,13870,693.500000
2023-01,46-60,27,62,12880,477.040000
2023-01,61+,4,12,520,130.000000
2023-02,18-30,31,81,18855,608.230000
2023-02,31-45,22,59,13195,599.770000
2023-02,46-60,25,62,10360,414.400000
2023-02,61+,7,12,1650,235.710000
2023-03,18-30,18,46,9660,536.670000
2023-03,31-45,21,58,7130,339.520000


unit_price_band,product_category,transaction_count,total_units,total_revenue,mean_transaction_value
Low,Clothing,215,544,19180,89.210000
Low,Electronics,203,497,17505,86.230000
Low,Beauty,186,460,16415,88.250000
Standard,Clothing,72,193,57900,804.170000
Standard,Electronics,72,183,54900,762.500000
Standard,Beauty,53,142,42600,803.770000
Premium,Beauty,68,169,84500,1242.650000
Premium,Electronics,67,169,84500,1261.190000
Premium,Clothing,64,157,78500,1226.560000


In [23]:
pivot = clean_df.pivot_table(
    index="age_group",
    columns="product_category",
    values="total_amount",
    aggfunc="sum",
    observed=False,
    dropna=False,
    fill_value=0,
)
pivot = pivot.reindex(
    index=AGE_LABELS,
    columns=["Beauty", "Clothing", "Electronics"],
    fill_value=0,
)
pivot.index.name = "Age group"

show_table(pivot, "Total revenue by age group and product category")

Age group,Beauty,Clothing,Electronics
18-30,45710,48670,38565
31-45,45215,47365,49375
46-60,47540,48415,51920
61+,5050,11130,17045


In [24]:
REPORT_NAME = "PS1_retail_category_gender_summary.csv"

export_df = summary_cat_gender.reset_index()
export_df.to_csv(REPORT_NAME, index=False)

report_info = pd.DataFrame(
    {
        "Value": [
            REPORT_NAME,
            os.path.abspath(REPORT_NAME),
            os.path.exists(REPORT_NAME),
            str(export_df.shape),
        ]
    },
    index=["File name", "Colab path", "File exists", "Shape"],
)
report_info.index.name = "Item"

show_table(report_info, "Exported report")
show_table(export_df.head(), "Exported report: first 5 rows")

Item,Value
File name,PS1_retail_category_gender_summary.csv
Colab path,/content/PS1_retail_category_gender_summary.csv
File exists,True
Shape,"(6, 7)"


index,product_category,gender,transaction_count,total_units,total_revenue,mean_transaction_value,median_transaction_value
0,Clothing,Female,174,441,81275,467.100000,120.000000
1,Electronics,Male,172,410,80170,466.100000,120.000000
2,Electronics,Female,170,439,76735,451.380000,150.000000
3,Beauty,Female,166,418,74830,450.780000,150.000000
4,Clothing,Male,177,453,74305,419.800000,150.000000


### Reading the extraction and aggregation results

- Extraction 1 returned 319 records. These are the larger multi-unit sales, above
  the median value of 135.00 and with quantity 3 or more.
- Extraction 2 returned 219 records for customers aged 31 to 45 buying Clothing or
  Electronics, sorted oldest first.
- Extraction 3 shows the final 10 transactions by date.
- The Q1 summary shows Clothing and Female with the highest total revenue, 81,275.
- The Q2 summary shows the strongest month and age group cell is 46-60 in 2023-05,
  with revenue 23,795.
- The Q3 summary shows the Premium band carries the most revenue, 247,500, even
  though it holds the fewest transactions of the three bands.
- In the pivot table the largest single cell is 46-60 buying Electronics at 51,920.
  The smallest is 61+ buying Beauty at 5,050.

---

## Task 5: Integrated Data Pipeline, Insights, and Conclusion

In [25]:
# rerun both functions as one repeatable pipeline
pipeline_raw = load_and_validate(
    dataset_handle    = SOURCE_METADATA["fixed_handle"],
    required_filename = SOURCE_METADATA["required_filename"],
    expected_shape    = EXPECTED_SHAPE,
    required_columns  = REQUIRED_COLUMNS,
)

pipeline_clean = clean_and_transform(
    df           = pipeline_raw,
    rename_map   = RENAME_MAP,
    age_bins     = AGE_BINS,
    age_labels   = AGE_LABELS,
    price_bins   = PRICE_BINS,
    price_labels = PRICE_LABELS,
)

flow = pd.DataFrame(
    {
        "Input":  ["Fixed Kaggle handle, version 1", f"raw_df {pipeline_raw.shape}"],
        "Step":   ["load_and_validate", "clean_and_transform"],
        "Output": [f"raw_df {pipeline_raw.shape}", f"clean_df {pipeline_clean.shape}"],
    },
    index=["Ingestion and validation", "Cleaning and transformation"],
)
flow.index.name = "Stage"

show_table(flow, "Pipeline: input, process, output")
print("Rerun gives the same shape as the original run:",
      pipeline_clean.shape == clean_df.shape)

Using Colab cache for faster access to the 'retail-sales-dataset' dataset.
INGESTION
  Handle used     : mohammadtalib786/retail-sales-dataset/versions/1
  Download folder : /kaggle/input/retail-sales-dataset
  Files available : ['retail_sales_dataset.csv', '.nfs000000007c0631d200000034']
  File loaded     : /kaggle/input/retail-sales-dataset/retail_sales_dataset.csv
  Raw header line : Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
  Delimiter       : ',' gives 9 header fields
  Records loaded  : 1000

VALIDATION
  Actual columns     : ['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']
  Missing columns    : None (0)
  Unexpected columns : None (0)
  Shape found        : (1000, 9)
  Shape expected     : (1000, 9)
  Shape matches      : True

  RESULT: VALIDATION PASSED
Rows with no age group  : 0
Rows with no price band : 0


Stage,Input,Step,Output
Ingestion and validation,"Fixed Kaggle handle, version 1",load_and_validate,"raw_df (1000, 9)"
Cleaning and transformation,"raw_df (1000, 9)",clean_and_transform,"clean_df (1000, 12)"


Rerun gives the same shape as the original run: True


In [26]:
# rows removed under each cleaning rule
removal = pd.DataFrame(
    {"Rows removed": [0, 0, 0, 0, 0, 0]},
    index=[
        "Missing values",
        "Exact duplicate rows",
        "Invalid Transaction ID",
        "Invalid Customer ID",
        "Invalid category label",
        "Out of range number",
    ],
)
removal.index.name = "Cleaning rule"

verification = pd.DataFrame(
    {
        "Value": [
            str(EXPECTED_SHAPE),
            str(clean_df.shape),
            int(removal["Rows removed"].sum()),
            "Not applicable, no rule removed any row",
            int(clean_df.isna().sum().sum()),
            int(clean_df.duplicated().sum()),
            all(c in clean_df.columns for c in RENAME_MAP.values()),
            all(c in clean_df.columns for c in NEW_COLUMNS),
            list(summary_cat_gender.columns) == list(AGG_WITH_MEDIAN.keys()),
            os.path.exists(REPORT_NAME),
        ]
    },
    index=[
        "Validated initial shape",
        "Final cleaned shape",
        "Total rows removed",
        "Overlap between removal rules",
        "Missing values remaining",
        "Duplicate rows remaining",
        "All required columns present",
        "All derived columns present",
        "Aggregation columns correct",
        "Exported file exists",
    ],
)
verification.index.name = "Item"

show_table(removal, "Rows removed under each cleaning rule")
show_table(verification, "Final verification")

Cleaning rule,Rows removed
Missing values,0
Exact duplicate rows,0
Invalid Transaction ID,0
Invalid Customer ID,0
Invalid category label,0
Out of range number,0


Item,Value
Validated initial shape,"(1000, 9)"
Final cleaned shape,"(1000, 12)"
Total rows removed,0
Overlap between removal rules,"Not applicable, no rule removed any row"
Missing values remaining,0
Duplicate rows remaining,0
All required columns present,True
All derived columns present,True
Aggregation columns correct,True
Exported file exists,True


In [27]:
# Q1: highest revenue combination, all ties reported
top_revenue = summary_cat_gender["total_revenue"].max()
q1_answer = summary_cat_gender[summary_cat_gender["total_revenue"] == top_revenue]

# Q2: strongest month and age group cells
q2_answer = summary_month_age.sort_values("total_revenue", ascending=False).head(5)

# Q3: revenue by price band
q3_answer = (
    summary_band_cat.groupby(level="unit_price_band", observed=True)[
        ["transaction_count", "total_units", "total_revenue"]
    ].sum()
)

show_table(q1_answer, "Q1 answer: highest revenue product category and gender")
show_table(q2_answer, "Q2 answer: top 5 month and age group cells by revenue")
show_table(q3_answer, "Q3 answer: totals for each unit price band")

print("Number of tied combinations for Q1 :", len(q1_answer))
print("Highest total revenue              :", top_revenue)

product_category,gender,transaction_count,total_units,total_revenue,mean_transaction_value,median_transaction_value
Clothing,Female,174,441,81275,467.100000,120.000000


month_period,age_group,transaction_count,total_units,total_revenue,mean_transaction_value
2023-05,46-60,38,103,23795,626.180000
2023-04,31-45,34,84,21295,626.320000
2023-12,31-45,30,73,19780,659.330000
2023-02,18-30,31,81,18855,608.230000
2023-08,46-60,36,93,18770,521.390000


unit_price_band,transaction_count,total_units,total_revenue
Low,604,1501,53100
Standard,197,518,155400
Premium,199,495,247500


Number of tied combinations for Q1 : 1
Highest total revenue              : 81275


### Answers to the three assigned questions

**Q1.** **Clothing and Female**, with total revenue **81,275**, **174** transactions, **441** units and mean transaction value **467.10**. Only **1** combination reached the maximum, so there was no tie to report. The margin is small: Electronics and Male follows at 80,170, a gap of 1.4 percent, and all six combinations fall between 68,685 and 81,275. Supporting output: the product_category-gender groupby summary (Task 4) and the Q1 answer table (Task 5).

**Q2.** The full breakdown is in the month_period-age_group summary. The strongest cell is **46-60 in 2023-05** with revenue **23,795** from 38 transactions, then 31-45 in 2023-04 at 21,295 and 31-45 in 2023-12 at 19,780. No age group leads in every month. 2024-01 holds only **2** transactions and is a partial month, so its low figures are where the data stops, not a real fall. Supporting output: the month_period-age_group summary (Task 4).

**Q3.** **Premium** carries the most revenue, **247,500** from **199** transactions and 495 units; Standard 155,400 from 197; Low 53,100 from 604. Count and value reverse: Low is 60.4 percent of transactions but 11.6 percent of revenue, Premium is 19.9 percent of transactions but 54.3 percent of revenue. Inside Premium, Beauty and Electronics tie at 84,500, Clothing 78,500. With only 5 distinct prices the bands are coarse. Supporting output: the unit_price_band-product_category summary (Task 4) and the Q3 band totals (Task 5).

### Five evidence supported findings

1. **Revenue is concentrated in expensive sales.** 199 Premium transactions give 247,500, 54.3 percent of the 456,000 total, while 604 Low transactions give 53,100. *(Q3 band totals table)*
2. **The mean alone is misleading.** Mean total_amount 456.00 against median 135.00, skewness 1.3741. *(Task 3 summary statistics and SciPy tables)*
3. **No category or gender segment dominates.** The six combinations span 68,685 to 81,275, an 18 percent range; gender totals are 232,840 Female and 223,160 Male. *(Q1 summary)*
4. **The 61+ group is small and uneven.** 93 of 1000 customers, spending 17,045 on Electronics but only 5,050 on Beauty. *(age_group-product_category pivot table)*
5. **No month on month trend can be claimed.** Monthly counts swing from 65 in 2023-09 to 105 in 2023-05 with no direction, and 2024-01 has only 2 transactions. *(Q2 summary)*

### Limitations

1. The dataset is synthetic, so it does not describe a real business.
2. Only 5 distinct unit prices, so the bands describe the price list, not a continuous range.
3. Age reaches only 64, so the 61+ band covers just 61-64.
4. 2024-01 is a partial month with 2 transactions and distorts any trend.
5. No cost or discount column, so revenue is not profit.
6. No store, region or channel column.
7. 1000 rows across 13 months and 4 age groups leaves under 40 transactions in most cells.
8. Every customer ID appears once, so repeat purchase cannot be studied.

### Conclusion

Revenue here is driven by price point, not customer segment: Premium supplies over half the revenue from a fifth of the transactions, while the six category-gender combinations differ by only 18 percent. The practical reading is to protect the small number of high value sales rather than target any one demographic. The dataset is synthetic with five price points and no repeat customers, so this is an exercise in method rather than a finding about a real market.
